In [1]:
# ════════════════════════════════════════════════════════════════════════════════════════════
# ScoutIA — VÉRIFICATION DU MODULE DE PRODUCTION analysis/montage_joueur.py — 19/11/2026
# Cette cellule NE CONTIENT PLUS le code du montage : elle clone le dépôt et appelle monter_joueur(), exactement comme
# le fera le pipeline. Comparer avec les runs des cellules (mêmes chiffres attendus, aux améliorations près) :
#   n° 6 rouge : présence ≈ 9,5 min, 31 moments avec le ballon, 13 buts du match dans le résumé ;
#   gardien n° 1 bleu (maillot noir) : ≈ 40 passages, 6 arrêts, 10 tirs adverses, 6 mètres retirés.
# ENTRÉES : codeprojectai (cache_sruverviers_reference_DETR, bout_en_bout_sruverviers.json, gemini_cache_sruverviers.jsonl,
#   lectures_v30_sruverviers_v2.json, parseq_scoutia_maillots_v2.pt), la vidéo SRU Verviers, mziparameter.
# GitHub : fichiers du dossier github_17_11 (dont analysis/montage_joueur.py et analysis/montage_joueur_script.py).
# Pas de GPU nécessaire (≈ 30 min par joueur sur processeur avec les lectures réutilisées). Save & Run All.
# ════════════════════════════════════════════════════════════════════════════════════════════
JOUEURS = [dict(numero=1, couleur="bleu", gardien=True, couleur_maillot="noir"),
           dict(numero=1, couleur="rouge", gardien=True, couleur_maillot="jaune")]

import glob, json, os, subprocess, sys

REPO = "/kaggle/working/ProjectAIMika"
if not os.path.isdir(REPO):
    if os.system(f"git clone -q --depth 1 https://github.com/mikarocco-droid/ProjectAIMika.git {REPO}") != 0:
        raise SystemExit("❌ clone du dépôt impossible")
if not os.path.exists(f"{REPO}/analysis/montage_joueur.py"):
    raise SystemExit("❌ analysis/montage_joueur.py absent de GitHub — pousser les fichiers du dossier github_17_11")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm", "pytorch-lightning", "nltk", "fpdf2", "pypdf",
                "google-genai", "opencv-python-headless"], check=False)
sys.path.insert(0, REPO)


def _un(motif, quoi):
    r = sorted(glob.glob(f"/kaggle/input/**/{motif}", recursive=True))
    if not r:
        raise SystemExit(f"❌ {quoi} introuvable dans les entrées")
    return r[0]


relais = json.load(open(_un("bout_en_bout_sruverviers.json", "bout_en_bout_sruverviers.json"), encoding="utf-8"))
video = _un("p4-sru-verviers-b-2026-10-10.mp4", "la vidéo SRU Verviers")
cache = os.path.dirname(_un("cache_sruverviers_reference_DETR/frames_data.pkl", "cache_sruverviers_reference_DETR"))
for f_ in ("lectures_v30_sruverviers_v2.json", "gemini_cache_sruverviers.jsonl", "parseq_scoutia_maillots_v2.pt"):
    _un(f_, f_)
pe = next(iter(glob.glob("/kaggle/input/**/mziparameter/**/param.env", recursive=True)), None)
cle = ""
if pe:
    for l in open(pe, encoding="utf-8"):
        if l.strip().startswith("GEMINI_API_KEY="):
            cle = l.split("=", 1)[1].strip().strip('"').strip("'")

from analysis.montage_joueur import monter_joueur

for j in JOUEURS:
    sortie = f"/kaggle/working/joueur_{j['numero']}_{j['couleur']}"
    print(f"\n{'═' * 90}\nJOUEUR n°{j['numero']} {j['couleur']}{' (gardien)' if j.get('gardien') else ''}\n{'═' * 90}", flush=True)
    res = monter_joueur(video=video, cache_dir=cache, sortie=sortie, match="sruverviers", bornes=relais.get("bornes"),
                        buts=relais.get("buts"), entrees=["/kaggle/input"], gemini_key=cle, **j)
    for k in ("trouve", "erreur", "message", "presence_s", "passages", "moments_ballon", "buts_confirmes",
              "buts_possibles", "arrets", "tirs_adverses", "cout_gemini_usd", "duree_min", "montage", "fiche",
              "rapport_et_fiche", "journal"):
        if k in res:
            print(f"   {k:18s} : {res[k]}")
print("\nÀ garder : les dossiers /kaggle/working/joueur_* (montage, fiche, journal_montage_joueur.txt).")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.6/360.6 kB 7.3 MB/s eta 0:00:00

══════════════════════════════════════════════════════════════════════════════════════════
JOUEUR n°1 bleu (gardien)
══════════════════════════════════════════════════════════════════════════════════════════
   relais : fourni par le pipeline
   BOUT EN BOUT : sruverviers | cache neuf /kaggle/input/datasets/michaelziant/codeprojectai/cache_sruverviers_reference_DETR | 0 action(s) du résumé produites par le pipeline
✅ Cache : /kaggle/input/datasets/michaelziant/codeprojectai/cache_sruverviers_reference_DETR
   coup d'envoi 11:31 | 45770 événements | 33129 images suivies | 0 numéros lus
✅ time_since_update présent (100.0 % des positions) : seules les vraies détections (== 0) sont utilisées
   4956 pistes DeepSort → 34056 segments | coupures : {'trou': 29078, 'saut': 2, 'taille': 20} | positions prédites écartées : 1114656 | hors 